# Optional: Text and Image Learning

**Working Example.** Copy this file, rename it, and modify your copy.
Do not edit this example; it should stay runnable and available as a reference
until after your work has been assessed.

- Author: Denise Case
- Date: 2026-06
- Dataset: Seaborn Penguins
- Target: body_mass_g

Run all cells top to bottom (**Run All**) before pushing to GitHub.

## M7. 

> This notebook runs the same train/evaluate skeleton on **text** and on **images**.
> The new idea is the **representation**: turning words and pixels into numbers. What
> each representation throws away is **your** judgment to name.

**Module 7 objectives** (course objectives in parentheses):
1. Apply and evaluate models for text-based learning. *(2, 3, 4)*
2. Apply and evaluate models for image-based learning. *(2, 3, 4)*

Both datasets are self-contained (no downloads): a small labeled text corpus
written inline, and scikit-learn's bundled handwritten-digit images.

## A. Prepare the Project Environment (.venv/)

- Open one project in VS Code at a time.
- Prepare the .venv/: specify Python version and install / upgrade dependencies listed in `pyproject.toml`.
- Open an integrated terminal (PowerShell if Windows) in the **root project** folder and run:

```shell
uv self update
uv python pin 3.14
uv lock --upgrade
uv sync --extra dev --extra docs --upgrade
```


## B. Select the Notebook Kernel

- Click on the **Select Kernel** name in the top-right corner of the notebook interface.
- Choose Python Environments... /
- Choose the recommended local .venv/ from the drop-down menu.
- This will create a new kernel for the notebook and allow the notebook to use packages installed in the .venv/ environment.

## C. Working in Notebooks (Custom Notes)

- To run a cell, press **Ctrl+Enter** (or **Cmd+Enter** on Mac) when done editing the cell.
- Change the type of a cell (e.g., code or markdown) by looking in the lower left corner of the notebook interface.
- Rearrange cells by dragging and dropping them within the notebook.

See [Run Jupyter Notebooks](https://denisecase.github.io/pro-analytics-02/workflow-b-apply-example-project/run-notebook/) for:

- how to **copy a notebook**
- how to release a `project.log` file
- how to deal with a **stuck kernel**
- etc.


## M7. Text and Images as Data

Models work on numbers. Text and images are not numbers until you **represent** them
as numbers - and once you do, it is the same classify-and-evaluate work as Module 3.

- **Text -> numbers** with **TF-IDF**: each document becomes a vector of word weights,
  high for words that are frequent in this document but rare across the corpus. This
  captures *which words* appear, but throws away **word order and meaning** ("dog bites
  man" and "man bites dog" look identical).
- **Images -> numbers** by reading **pixel values**. An 8x8 grayscale image is 64
  numbers. Flattening to a feature vector throws away **spatial structure** (which
  pixels are neighbors).

Naming what a representation discards is the judgment this module adds. The modeling
and evaluation are the skills you already have.

## Section 1. Project Setup and Imports

In [2]:
# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify

from datafun_toolkit.logger import get_logger, log_header
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M07", level="DEBUG")
log_header(LOG, "M07")

RANDOM_STATE: int = 123


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")
LOG.info(f"  numpy:        {version('numpy')}")
LOG.info(f"  scikit-learn: {version('scikit-learn')}")
LOG.info(f"  seaborn:      {version('seaborn')}")
LOG.info(f"  matplotlib:   {version('matplotlib')}")

# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

2026-06-28 15:06:54 | INFO | M07 | === RUN START ===
2026-06-28 15:06:54 | INFO | M07 | project=M07
2026-06-28 15:06:54 | INFO | M07 | repo_dir=ml-07-applied
2026-06-28 15:06:54 | INFO | M07 | python=3.14.0
2026-06-28 15:06:54 | INFO | M07 | os=Windows 11
2026-06-28 15:06:54 | INFO | M07 | shell=powershell
2026-06-28 15:06:54 | INFO | M07 | cwd=notebooks
2026-06-28 15:06:54 | INFO | M07 | github_actions=False
2026-06-28 15:06:54 | INFO | M07 | Confirming installation:
2026-06-28 15:06:54 | INFO | M07 |   python:       3.14.0
2026-06-28 15:06:54 | INFO | M07 |   pandas:       3.0.4
2026-06-28 15:06:54 | INFO | M07 |   numpy:        2.5.0
2026-06-28 15:06:54 | INFO | M07 |   scikit-learn: 1.9.0
2026-06-28 15:06:54 | INFO | M07 |   seaborn:      0.13.2
2026-06-28 15:06:54 | INFO | M07 |   matplotlib:   3.11.0


## Section 2. Text-Based Learning

```text
ANALYST CHOICE
TF-IDF turns each document into word weights. 
It captures which words appear, not their order or meaning. 
Decide whether that is good enough for your text problem.
```

In [3]:
# === Section 2. Text-Based Learning ===

# A small, self-contained labeled corpus (topic classification).
# CUSTOM: replace with your own labeled text in the Apply/Extend section.
TEXT_DOCS: list[str] = [
    "the team scored a goal and won the match",
    "he hit a home run to win the game",
    "the coach praised the players after the victory",
    "she sprinted the final lap and took the medal",
    "the tournament final goes to overtime tonight",
    "fans cheered as the striker scored again",
    "the quarterback threw a long touchdown pass",
    "they trained hard before the championship game",
    "preheat the oven and bake the bread for an hour",
    "add salt and stir the simmering tomato sauce",
    "chop the onions and saute them in olive oil",
    "the recipe needs flour, butter, sugar, and eggs",
    "let the dough rise before you knead it again",
    "grill the vegetables and season with pepper",
    "whisk the eggs and fold in the melted chocolate",
    "simmer the soup and serve it warm with crusty bread",
    "the telescope captured a distant spiral galaxy",
    "the rocket launched into orbit at dawn",
    "astronauts repaired the satellite during the spacewalk",
    "the probe sent back images of the red planet",
    "a comet will pass near earth this winter",
    "the observatory tracked a newly discovered asteroid",
    "the mission will land a rover on the lunar surface",
    "scientists measured radiation from a nearby star",
]
TEXT_LABELS: list[str] = ["sports"] * 8 + ["cooking"] * 8 + ["space"] * 8


def run_text_pipeline() -> None:
    """Represent text with TF-IDF, classify it, and evaluate.

    WHY: Once text is TF-IDF vectors, this is the same fit/predict/score work
    as Module 3. The representation is the only new part.

    Returns:
        None
    """
    LOG.info(
        f"Text corpus: {len(TEXT_DOCS)} documents, classes={sorted(set(TEXT_LABELS))}"
    )

    docs_train, docs_test, y_train, y_test = train_test_split(
        TEXT_DOCS,
        TEXT_LABELS,
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=TEXT_LABELS,
    )

    # TF-IDF: fit the vocabulary on TRAIN only, then transform both.
    vectorizer = TfidfVectorizer()
    X_train = vectorizer.fit_transform(docs_train)
    X_test = vectorizer.transform(docs_test)
    LOG.info(f"Vocabulary size (features): {len(vectorizer.vocabulary_)}")

    model = MultinomialNB()
    model.fit(X_train, y_train)

    acc: float = float(accuracy_score(y_test, model.predict(X_test)))
    LOG.info(f"Text test accuracy: {acc:.3f}")

    # Predict a couple of brand-new sentences.
    new_docs: list[str] = [
        "the midfielder scored in the last minute of the game",
        "stir the sauce and bake the casserole until golden",
    ]
    preds = model.predict(vectorizer.transform(new_docs))
    for doc, pred in zip(new_docs, preds, strict=True):
        LOG.info(f"  predicted '{pred}'  <-  {doc!r}")

    LOG.info("TF-IDF ignored word order. Did any misread sentence reveal that limit?")

## Section 3. Image-Based Learning

```text
ANALYST CHOICE
Flattening an 8x8 image to 64 pixel features discards which pixels are neighbors.
A model still learns digits from raw pixels; decide where that representation
would break down (rotation, scale, position).
```

In [4]:
from typing import Any, cast

# === Section 3. Image-Based Learning ===


def run_image_pipeline() -> None:
    """Represent images as pixel vectors, classify the digits, and evaluate.

    WHY: An 8x8 grayscale image is 64 numbers. After flattening, it is again the
    same fit/predict/score work. We also look at a few images so the data is not
    just an abstract matrix.

    Returns:
        None
    """
    digits = load_digits()
    digits = cast(Any, load_digits())
    LOG.info(
        f"Image data: {digits.data.shape[0]} images, {digits.data.shape[1]} pixel features each"
    )
    LOG.info(f"Classes (digits): {sorted(set(digits.target.tolist()))}")

    # Look at the data: show the first few images with their true labels.
    fig, axes = plt.subplots(1, 5, figsize=(8, 2))
    for ax, image, label in zip(axes, digits.images, digits.target, strict=False):
        ax.imshow(image, cmap="gray")
        ax.set_title(f"label {label}")
        ax.axis("off")
    plt.suptitle("A few input images (8x8 pixels)")
    plt.show()

    X = digits.data
    y = digits.target
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)

    acc: float = float(accuracy_score(y_test, model.predict(X_test)))
    LOG.info(f"Image test accuracy: {acc:.3f}")

    LOG.info("Showing confusion matrix (which digits get confused)")
    plt.figure(figsize=(6, 6))
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
    plt.title("Digit confusion matrix (test set)")
    plt.show()

    LOG.info("Which digits confuse each other, and does the pixel view explain why?")

## Section 4. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [5]:
# === Python Summary ===


def summarize() -> None:
    """Record what was applied and the judgment still owed."""
    LOG.info("========================")
    LOG.info("SUMMARY")
    LOG.info("========================")
    LOG.info("Text:   TF-IDF + MultinomialNB on a labeled corpus")
    LOG.info("Image:  pixel features + LogisticRegression on digit images")
    LOG.info("Same skeleton both times; only the representation changed.")
    LOG.info("========================")
    LOG.info("YOUR JUDGMENT (write up in README.md and docs/index.md):")
    LOG.info("  - What did TF-IDF throw away, and where would that bite you?")
    LOG.info(
        "  - What did flattening images throw away, and where would that bite you?"
    )
    LOG.info("  - For each, what representation would you reach for next, and why?")
    LOG.info("========================")


### Narrative

Summarize your work in a Markdown cell in the notebook.

### Conclusions

This is SUPERVISED learning (because the data includes a target).

The chosen target is `species` and that is a category column. 

Therefore, modeling this problem will use classification.

### Next Steps

Summarize your next steps in a Markdown cell in the notebook.


## Task: Make the Notebook Yours (Apply / Extend / Explore)

This is an example.
Copy this notebook and make it your own. 

In your copy:

1. At the beginning, update the Author, the purpose, the target, etc. 
2. Remove any instructions you do not need. 

Try things like the following.

1. **Apply (text)** - Replace `TEXT_DOCS`/`TEXT_LABELS` with your own labeled
   sentences (your domain) and re-run. Report where the model misreads and whether
   word order was the cause.
2. **Apply (image)** - Swap `LogisticRegression` for a `RandomForestClassifier` on
   the digits and compare. Show a few misclassified images and reason about why.
3. **Explore** - Add `ngram_range=(1, 2)` to `TfidfVectorizer` so it sees word pairs.
   Decide in `docs/index.md` whether recovering a little word order helped your text task.

**Assessed:** `README.md` and `docs/index.md` - your representations, your scores,
and your account of what each representation discards. Keep this example alongside
your work until assessed.

## Task: Final Check

- `README.md` - reflects your description, instructions, commands, and links to your executed notebook.
- `docs/index.md` - reflects your project-specific updates.
- Your GitHub **About** section has a link to your hosted documentation site.
- The executed example notebook AND your custom notebook are available in `notebooks/`.
- Keep this **working example** alongside your custom work until your work has been assessed.
- Ensure your **custom notebook** introduces and narrates **your** custom project.

## Reminder: Run All before pushing to GitHub

Before saving a notebook (and running git add-commit-push), click 'Run All' to generate all outputs and display them in the notebook.

Follow our [pro-analytics-02](https://denisecase.github.io/pro-analytics-02/) common workflows.

Your README.md should have a description, a link to your executed notebook, and a list of commands (updated as you add your custom description, instructions, and commands).

Your docs/ folder should document your custom project analysis in the `docs/index.md` summary.
